In [ ]:
import sys
import os

# Adjust the path to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import matplotlib.ticker as ticker
from sklearn.metrics import mean_squared_error

df_gas = pd.read_csv('../data/Gas_polygon.csv')
df_selected = pd.read_csv('../data/qgis_polygon.csv')

# Problematic ids
- 5 needs to be sum to 6
- 40 and 97 don't have a position equivalent on QGIS
- 107 and 115 have equivalent on QGIS, but heat value is NaN
- 109 was fixed manually in QGIS
- 42 is also problematic, even though it has an equivalent position in syntetic heat data.

In [ ]:
row_5 = df_gas[df_gas['id'] == 5].iloc[0]
row_6 = df_gas[df_gas['id'] == 6].iloc[0]

combined_row = row_5.copy()
years_array = np.arange(2012, 2024)
for year in years_array:
    year = str(year)
    combined_row[year] = row_5[year] + row_6[year]  

combined_row['xcoord'] = row_5['xcoord']
combined_row['ycoord'] = row_5['ycoord']

df_gas = df_gas[df_gas['id'] != 5]
df_gas = df_gas[df_gas['id'] != 6]

df_gas = pd.concat([df_gas, pd.DataFrame([combined_row])], ignore_index=True)

problematic_ids = [40, 97, 107, 115]
for id in problematic_ids:
    df_gas = df_gas[df_gas['id'] != id]

df_gas.reset_index(drop=True, inplace=True)

In [ ]:
display(len(df_selected))
display(len(df_gas))

# Merge datasets for comparison

In [ ]:
merged_df = pd.merge(df_gas, df_selected, left_on=['id_polygon'],
                     right_on=['id_polygon'], how='inner')

# Check if the number of rows in merged_df is equal to the number of rows in df_gas
if len(merged_df) == len(df_gas):
    print("All rows in gas data have a corresponding row in heat data.")
else:
    print(f"Not all rows in gas data have a corresponding row in heat data. Only {len(merged_df)} found a match.")

In [ ]:
merged_df.to_csv('../data/merged_data.csv')

# Calculate actual energy used with gas consumption
- Efficiency of gas boilers depends on the age of the building?
- Can we correlate the age of buildings to age of gas boilers?
- We consider [90%](https://www.greenmatch.co.uk/blog/2015/09/how-efficient-is-a-condensing-boiler) efficiency 

In [ ]:
efficiency_factor = 0.9
for year in range(2012, 2024):
    column_name = str(year)
    if column_name in merged_df.columns:
        merged_df[column_name] *= efficiency_factor

In [ ]:
plt.plot(merged_df.index, merged_df['qnutzwaerme_2020_kwh'], label='Synthetic data')
plt.plot(merged_df.index, merged_df['2020'], label='Real data')
formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))

# Apply the formatter to the x and y axes
plt.gca().yaxis.set_major_formatter(formatter)

# Ensure the scientific notation (xEy) is displayed on the top
plt.gca().ticklabel_format(style='scientific', axis='both', useOffset=True)
plt.xlabel('Index')
plt.ylabel('Energy consumption (kWh)')
plt.legend()
plt.show()

In [ ]:
print(f'Mean of real data: {round(merged_df["2020"].mean(), 2)}kWh')
print(f'Mean of synthetic data: {round(merged_df["qnutzwaerme_2020_kwh"].mean(),2)}kWh')

In [ ]:
correlation = merged_df['qnutzwaerme_2020_kwh'].corr(merged_df['2020'])
print(f"Correlation coefficient between qnutzwaerme_2020_kwh and 2020: {correlation}")

plt.figure(figsize=(10, 6))
sns.scatterplot(x='qnutzwaerme_2020_kwh', y='2020', data=merged_df)
formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))

# Apply the formatter to the x and y axes
plt.gca().xaxis.set_major_formatter(formatter)
plt.gca().yaxis.set_major_formatter(formatter)

# Ensure the scientific notation (xEy) is displayed on the top
plt.gca().ticklabel_format(style='scientific', axis='both', useOffset=True)

plt.title(f"Real x Synthetic Energy Demand for Heating in 2020 (Correlation: {correlation:.2f})")
plt.xlabel('Synthetic energy demand in 2020 (kWh)')
plt.ylabel('Energy demand from real data in 2020 (kWh)')
plt.show()

In [ ]:
font_size_title = 16
font_size_labels = 14
font_size_ticks = 12

sns.histplot(merged_df[['qnutzwaerme_2020_kwh', '2020']], kde=True, color='blue', label='Real data')
plt.xlabel('Energy consumption (kWh)', fontsize=font_size_labels)
plt.ylabel('Frequency', fontsize=font_size_labels)
plt.legend(['Processed data', 'Synthetic data'], fontsize=font_size_labels)
plt.xticks(fontsize=font_size_ticks)
plt.yticks(fontsize=font_size_ticks)
plt.title('Energy Consumption Distribution', fontsize=font_size_title)  # Optional title, adjust as needed

In [ ]:

boxplot = sns.boxplot(data=merged_df[['qnutzwaerme_2020_kwh', '2020']])

# Set the y-axis label with font size
boxplot.set_ylabel('Energy demand (kWh)', fontsize=font_size_labels)

# Set the x-tick labels with font size
boxplot.set_xticklabels(['Synthetic', 'Processed'], fontsize=font_size_labels)

# Optional title with font size
plt.title('Energy Demand Distribution', fontsize=font_size_title)

# Set the font size for the x and y ticks
plt.xticks(fontsize=font_size_ticks)
plt.yticks(fontsize=font_size_ticks)

In [ ]:
quantiles_2020_full = merged_df['2020'].quantile([0, 0.25, 0.5, 0.75, 1])
quantiles_qnutzwaerme_full = merged_df['qnutzwaerme_2020_kwh'].quantile([0, 0.25, 0.5, 0.75, 1])

# Create a table comparing quantiles (min, 25%, 50%, 75%, max)
quantile_comparison_full = pd.DataFrame({
    '2020 (Gas consumption)': quantiles_2020_full,
    'qnutzwaerme_2020_kwh (Synthetic heat)': quantiles_qnutzwaerme_full
})

# Print the means separately
mean_2020 = merged_df['2020'].mean()
mean_qnutzwaerme = merged_df['qnutzwaerme_2020_kwh'].mean()

# Prepare LaTeX format for Overleaf
latex_table = quantile_comparison_full.to_latex(index=False)

# Print the quantile comparison and means
print("Quantile Comparison (min, 25%, 50%, 75%, max):")
print(quantile_comparison_full)
print(f"\nMean 2020 (Gas consumption): {round(mean_2020,2)}")
print(f"Mean qnutzwaerme_2020_kwh (Synthetic heat): {round(mean_qnutzwaerme,2)}")
max_index = merged_df['2020'].idxmax()
print("Value of swimming pool in 2020: ", merged_df['2020'][max_index])
print("Synthetic value of swimming pool in 2020: ", merged_df['qnutzwaerme_2020_kwh'][max_index])

# Print LaTeX code for Overleaf
print("\nLaTeX Table Format for Overleaf:")
print(latex_table)

In [ ]:
Q1 = np.percentile(merged_df['2020'], 25)
Q3 = np.percentile(merged_df['2020'], 75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Clip the values above the upper bound
merged_df['2020'] = np.where(merged_df['2020'] > upper_bound, upper_bound, merged_df['2020'])
merged_df['qnutzwaerme_2020_kwh'] = np.where(merged_df['qnutzwaerme_2020_kwh'] > upper_bound, upper_bound, merged_df['qnutzwaerme_2020_kwh'])

# Print the clipped value for reference
print(f'Clipped values from 0 to {round(upper_bound, 2)}')

# Create the boxplot
boxplot = sns.boxplot(data=merged_df[['qnutzwaerme_2020_kwh', '2020']])

# Set the y-axis label and font size
boxplot.set_ylabel('Energy demand (kWh)', fontsize=font_size_labels)

# Set the x-tick labels and font size
boxplot.set_xticklabels(['Synthetic', 'Processed'], fontsize=font_size_labels)

# Optional: Set the title and font size (if required)
plt.title('Energy Demand Distribution', fontsize=font_size_title)

# Set the font size for ticks
plt.xticks(fontsize=font_size_ticks)
plt.yticks(fontsize=font_size_ticks)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
quantiles_2020_full = merged_df['2020'].quantile([0, 0.25, 0.5, 0.75, 1])
quantiles_qnutzwaerme_full = merged_df['qnutzwaerme_2020_kwh'].quantile([0, 0.25, 0.5, 0.75, 1])

# Create a table comparing quantiles (min, 25%, 50%, 75%, max)
quantile_comparison_full = pd.DataFrame({
    '2020 (Gas consumption)': quantiles_2020_full,
    'qnutzwaerme_2020_kwh (Synthetic heat)': quantiles_qnutzwaerme_full
})

# Print the means separately
mean_2020 = merged_df['2020'].mean()
mean_qnutzwaerme = merged_df['qnutzwaerme_2020_kwh'].mean()

# Prepare LaTeX format for Overleaf
latex_table = quantile_comparison_full.to_latex(index=False)

# Print the quantile comparison and means
print("Quantile Comparison (min, 25%, 50%, 75%, max):")
print(quantile_comparison_full)
print(f"\nMean 2020 (Gas consumption): {round(merged_df["2020"].mean(), 2)}")
print(f"Mean qnutzwaerme_2020_kwh (Synthetic heat): {round(mean_qnutzwaerme,2)}")

# Print LaTeX code for Overleaf
print("\nLaTeX Table Format for Overleaf:")
print(latex_table)

In [ ]:
sns.histplot(merged_df[['qnutzwaerme_2020_kwh', '2020']], kde=True, color='blue')

# Set labels and font sizes
plt.xlabel('Energy consumption (kWh)', fontsize=font_size_labels)
plt.ylabel('Frequency', fontsize=font_size_labels)

# Set the legend and font size
plt.legend(['Processed data', 'Synthetic data'], fontsize=font_size_labels)

# Set font size for ticks
plt.xticks(fontsize=font_size_ticks)
plt.yticks(fontsize=font_size_ticks)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
print(f'Mean of clipped real data: {round(merged_df["2020"].mean(), 2)}kWh')
print(f'Mean of clipped synthetic data: {round(merged_df["qnutzwaerme_2020_kwh"].mean(),2)}kWh')
print(len(merged_df[merged_df["2020"] == merged_df["qnutzwaerme_2020_kwh"]]))

In [ ]:
merged_df.to_csv('../data/merged_data.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np

# Calculate the residuals: the difference between actual y-values and the y=x line
residuals = merged_df['2020'] - merged_df['qnutzwaerme_2020_kwh']

# Calculate the standard deviation of the residuals
std_residuals = np.std(residuals)

# Identify outliers: points where the absolute value of the residuals is greater than 3 times the standard deviation
outliers = np.abs(residuals) > 3 * std_residuals

# Print the formula of the y = x line
print("Formula of the fitted line: y = x")

plt.figure(figsize=(10, 6))

# Scatter plot of the original data
plt.scatter(merged_df['qnutzwaerme_2020_kwh'], merged_df['2020'], label='Data Points')

# Plot the y = x line
max_val = max(merged_df['qnutzwaerme_2020_kwh'].max(), merged_df['2020'].max())
min_val = min(merged_df['qnutzwaerme_2020_kwh'].min(), merged_df['2020'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='green', linestyle='--', label='y = x Line')

formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))

# Apply the formatter to the x and y axes
plt.gca().xaxis.set_major_formatter(formatter)
plt.gca().yaxis.set_major_formatter(formatter)

# Ensure the scientific notation (xEy) is displayed on the top
plt.gca().ticklabel_format(style='scientific', axis='both', useOffset=True)

# Highlight the outliers (optional)
# plt.scatter(merged_df['qnutzwaerme_2020_kwh'][outliers], merged_df['2020'][outliers], color='red', label='Outliers (> 3 std)', edgecolor='black')

# Labels and title with custom font sizes
plt.xlabel('Synthetic heat consumption in 2020 (kWh)', fontsize=14)
plt.ylabel('Processed gas consumption in 2020 (kWh)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True)

# Set tick label font sizes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Show the plot
plt.show()


In [ ]:
merged_df[['id_y', '2020','qnutzwaerme_2020_kwh']][outliers]

In [ ]:
nan_rows = merged_df[merged_df['qnutzwaerme_2020_kwh'].isna()]
print(nan_rows)

# Analysing outliers
- 25
- 38
- 106 

# Correlate variables from real and synthetic data

In [ ]:
merged_df_gas_heat = merged_df[['2020', '2021', '2022', '2023',
                     'qh_2020_kwh', 'qnutzwaerme_2020_kwh']]

In [ ]:
new_column_names = {
    '2020': 'gas_2020',
    '2021': 'gas_2021',
    '2022': 'gas_2022',
    '2023': 'gas_2023',
    'qh_2020_kwh': 'heat_2020',
    'qnutzwaerme_2020_kwh': 'heat_used_2020'
}
merged_df_gas_heat.rename(columns=new_column_names, inplace=True)

In [ ]:
correlation_matrix = merged_df_gas_heat.corr()

plt.figure(figsize=(12, 8))

sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='flare', linewidths=0.5, annot_kws={"size": 10})

plt.title('Heatmap of Correlations Between Columns in merged_df', fontsize=16)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

In [ ]:
gas_columns = [col for col in merged_df_gas_heat.columns if col.startswith('gas')]
heat_columns = [col for col in merged_df_gas_heat.columns if col.startswith('heat')]

# Create an empty DataFrame to store RMSE values
rmse_matrix = pd.DataFrame(index=gas_columns, columns=heat_columns, dtype=float)

# Compute RMSE for each pair of gas and heat columns
for gas_col in gas_columns:
    for heat_col in heat_columns:
        # Drop NA values to ensure both columns have the same length
        valid_indices = merged_df_gas_heat[[gas_col, heat_col]].dropna().index
        rmse = np.sqrt(mean_squared_error(merged_df_gas_heat.loc[valid_indices, gas_col],
                              merged_df_gas_heat.loc[valid_indices, heat_col]))
        rmse_matrix.loc[gas_col, heat_col] = rmse
plt.figure(figsize=(12, 8))
sns.heatmap(rmse_matrix.astype(float), annot=True, fmt=".2f", cmap='viridis', linewidths=0.5, annot_kws={"size": 10})

plt.title('Heatmap of RMSE Between Gas and Heat Columns', fontsize=16)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel('Heat Columns')
plt.ylabel('Gas Columns')
plt.show()

# Statistical metrics to compare data

In [ ]:
mse = mean_squared_error(merged_df['2020'], merged_df['qnutzwaerme_2020_kwh'])
rmse = np.sqrt(mse)
# Calculate the mean of the actual values (assuming '2020' column holds the actual values)
mean_actual = np.mean(merged_df['2020'])

# Calculate the percentage of RMSE on the mean
percentage_rmse = (rmse / mean_actual) * 100
print(f"Root Mean Squared Error (RMSE): {percentage_rmse:.2f}%")

In [ ]:
merged_df.to_csv('../data/merged_data.csv', index=False)